# ATLAS hydro CMIP6 preprocessing

This notebook preprocesses the CMIP6 files downloaded with the CMIP6 download notebook.

The workflow is organised step by step for users who are not expert Python users. It starts from the downloaded global CMIP6 NetCDF files, cuts them around the selected country, optionally aggregates the data to daily means, and saves one processed NetCDF file for each experiment.

The expected input folder structure is:

`../data/cmip6/downloads/{VARIABLE}/global/{experiment}/{MODEL}/`

For example:

`../data/cmip6/downloads/rivo/global/ssp370/CNRM-ESM2-1/`


## Step 1. Import libraries

This step loads the Python libraries needed to read NetCDF files, manage spatial coordinates, crop the data and save the processed outputs.


In [1]:
from pathlib import Path

import geopandas as gpd
import numpy as np
import xarray as xr
import rioxarray  # noqa: F401. Required to use the .rio accessor on xarray objects.
from shapely.ops import unary_union


## Step 2. Define user parameters

Edit only this cell to change country, variable, model, experiments or folders.

The input path follows the same structure used in the CMIP6 download notebook:

`BASE_INPUT_DIR / VARIABLE / "global" / experiment / MODEL`


In [2]:
# ---------------------------------------------------------------------
# Main configuration
# ---------------------------------------------------------------------

# Country folder used in the project directories.
COUNTRY_FOLDER = "ecuador"

# Country name as written in the Natural Earth shapefile.
COUNTRY_NAME_IN_SHAPEFILE = "Ecuador"

# CMIP6 variable name.
# For river discharge, use 'rivo'.
VARIABLE = "rivo"

# Descriptive name used in the output folder.
VARIABLE_LONG_NAME = "river_discharge"

# CMIP6 model.
MODEL = "CNRM-ESM2-1"

# Experiments to preprocess.
# Use the same experiments downloaded in the CMIP6 download notebook.
EXPERIMENTS = [ "historical", 
                "ssp370",
                "ssp585",
              ]

# Optional temporal aggregation.
# Use 'daymean' to compute daily means, or None to keep the original temporal resolution.
AGGREGATION_MODE = "daymean"

# Input folder used by the CMIP6 download notebook.
# Expected structure:
# ../data/esgf_downloads/{MODEL}/{experiment}/{VARIABLE}/
BASE_INPUT_DIR = Path("../data/esgf_downloads")

# Output folder for processed CMIP6 data.
# Files will be saved in:
# ../data/processed/{VARIABLE_LONG_NAME}/{COUNTRY_FOLDER}/{MODEL}/{experiment}/
BASE_OUTPUT_DIR = Path("../data/processed")

# Temporary folder used to save the spatial subset before aggregation.
# This is useful for long time series because it avoids keeping all steps only in memory.
BASE_TEMP_DIR = BASE_OUTPUT_DIR#Path("../data/temp/cmip6_preprocessing")

# Country boundaries.
SHAPEFILE_PATH = Path("../world_map/ne_50m_admin_0_countries.shp")

# Bounding boxes used to crop the global CMIP6 data.
# Format: [north, west, south, east]
AREAS = {
    "bolivia": [-9.6, -69.8, -23.0, -57.4],
    "argentina": [-21.7, -73.6, -55.1, -53.5],
    "ecuador": [1.9, -92.0, -5.3, -75.1],
    "peru": [0.1, -81.5, -18.5, -68.5],
    "colombia": [15.9, -81.7, -5.1, -65.9],
}

AREA = AREAS[COUNTRY_FOLDER]

# Extra spatial buffer around the country bounding box, in degrees.
# This keeps neighbouring grid cells that may be useful for later interpolation or basin statistics.
SPATIAL_BUFFER_DEGREES = 2.5

# Set to False if you want to keep existing processed files.
OVERWRITE = True


## Step 3. Define helper functions

These functions keep the main workflow short and readable.

They standardise coordinate names, convert longitudes from 0-360 to -180-180 when needed, crop the global dataset around the country and save the result as NetCDF.


In [3]:
def get_period_for_experiment(experiment):
    """Return the preprocessing period for a CMIP6 experiment."""
    if experiment == "historical":
        return "1985-01", "2014-12"
    return "2015-01", "2100-12"


def standardise_coordinates(ds):
    """Rename CMIP6 coordinates to longitude and latitude when needed."""
    rename_dict = {}
    if "lon" in ds.coords:
        rename_dict["lon"] = "longitude"
    if "lat" in ds.coords:
        rename_dict["lat"] = "latitude"
    if rename_dict:
        ds = ds.rename(rename_dict)
    return ds


def roll_longitudes(ds, lon_name="longitude"):
    """Convert longitudes from 0-360 degrees to -180-180 degrees."""
    ds = ds.assign_coords({
        lon_name: ((ds[lon_name] + 180) % 360) - 180
    })
    return ds.sortby(lon_name)


def ensure_epsg4326(ds):
    """Assign EPSG:4326 to datasets with longitude and latitude coordinates."""
    ds = ds.rio.set_spatial_dims(x_dim="longitude", y_dim="latitude", inplace=False)
    if ds.rio.crs is None:
        ds = ds.rio.write_crs("EPSG:4326", inplace=False)
    return ds


def fix_coordinates(ds):
    """Prepare coordinates for spatial selection."""
    ds = standardise_coordinates(ds)
    ds["longitude"] = np.round(ds.longitude, 3)
    ds["latitude"] = np.round(ds.latitude, 3)
    if ds.longitude.values.min() >= 0:
        ds = roll_longitudes(ds)
    ds = ensure_epsg4326(ds)
    return ds


def preprocess_single_file(ds, experiment):
    """Apply basic preprocessing while opening each CMIP6 file."""
    ds = fix_coordinates(ds)
    for optional_var in ["time_bounds", "time_bnds", "height"]:
        if optional_var in ds:
            ds = ds.drop_vars(optional_var)
    start, end = get_period_for_experiment(experiment)
    ds = ds.sel(time=slice(start, end))
    for dim_name in ["member_id", "dcpp_init_year"]:
        if dim_name in ds.dims:
            ds = ds.mean(dim_name)
    return ds


def load_cmip6_data(input_dir, experiment):
    """Open all NetCDF files found in one CMIP6 experiment folder."""
    input_files = sorted(input_dir.glob("*.nc"))
    if not input_files:
        raise FileNotFoundError(
            f"No NetCDF files found in {input_dir}. "
            "Check that the CMIP6 download notebook saved files in this folder."
        )
    print(f"Input files found: {len(input_files)}")
    print(f"First file: {input_files[0].name}")
    ds = xr.open_mfdataset(
        input_files,
        preprocess=lambda x: preprocess_single_file(x, experiment),
        combine="by_coords",
    )
    return ds


def cut_dataset_to_area(ds, area, buffer_degrees=2.5):
    """Crop the dataset around the country bounding box."""
    north, west, south, east = area
    lon_min = west - buffer_degrees
    lon_max = east + buffer_degrees
    lat_min = south - buffer_degrees
    lat_max = north + buffer_degrees
    ds = ds.sortby("latitude")
    return ds.sel(longitude=slice(lon_min, lon_max), latitude=slice(lat_min, lat_max))


def aggregate_daily_mean(ds, variable_name):
    """Compute daily mean values."""
    if "valid_time" in ds.coords:
        ds = ds.rename({"valid_time": "time"})
    return ds.resample(time="1D").mean()


def read_country_geometry(shapefile_path, country_name):
    """Read the selected country geometry from the Natural Earth shapefile."""
    countries = gpd.read_file(shapefile_path)
    country = countries[countries["NAME_EN"] == country_name]
    if country.empty:
        raise ValueError(
            f"Country '{country_name}' not found in {shapefile_path}. "
            "Check COUNTRY_NAME_IN_SHAPEFILE."
        )
    return gpd.GeoSeries([unary_union(country.geometry)], crs=country.crs)


def save_netcdf(ds, output_filename, overwrite=True):
    """Save an xarray Dataset or DataArray to NetCDF."""
    output_filename = Path(output_filename)
    output_filename.parent.mkdir(parents=True, exist_ok=True)
    if output_filename.exists() and not overwrite:
        print(f"File already exists and overwrite is False: {output_filename}")
        return
    if isinstance(ds, xr.DataArray):
        ds = ds.to_dataset()
    if "spatial_ref" in ds:
        ds = ds.drop_vars("spatial_ref")
    encoding = {}
    for data_var in ds.data_vars:
        variable_encoding = {"zlib": False}
        if np.issubdtype(ds[data_var].dtype, np.floating):
            variable_encoding["dtype"] = "float32"
        shape = ds[data_var].shape
        if ds[data_var].ndim == 3:
            variable_encoding["chunksizes"] = (1, min(shape[1], 512), min(shape[2], 512))
        elif ds[data_var].ndim == 2:
            variable_encoding["chunksizes"] = (min(shape[0], 512), min(shape[1], 512))
        encoding[data_var] = variable_encoding
    ds.to_netcdf(
        output_filename,
        engine="netcdf4",
        format="NETCDF4",
        encoding=encoding,
        compute=True,
    )
    print(f"Data written to: {output_filename}")


## Step 4. Check the country geometry

This step checks that the selected country can be found in the shapefile.

The geometry is not used to mask the CMIP6 data in this notebook. Here the data are cropped using a bounding box, because CMIP6 grid cells are coarse and the following basin statistics workflow may need cells just outside the administrative boundary.


In [4]:
country_geometry = read_country_geometry(
    shapefile_path=SHAPEFILE_PATH,
    country_name=COUNTRY_NAME_IN_SHAPEFILE,
)

print("Country geometry loaded correctly:")
print(COUNTRY_NAME_IN_SHAPEFILE)
print(country_geometry.total_bounds)


ERROR 1: PROJ: proj_create_from_database: Open of /home/alessandrom/anaconda3/envs/preprocess_conda/share/proj failed


Country geometry loaded correctly:
Bolivia
[-69.64570312 -22.89169922 -57.4956543   -9.71044922]


## Step 5. Preprocess each experiment

For each experiment, the notebook:

1. reads the NetCDF files from the CMIP6 download folder
2. standardises latitude and longitude names
3. converts longitude coordinates if necessary
4. selects the correct time period
5. crops the data around the selected country
6. optionally aggregates the data to daily means
7. saves the processed NetCDF file


In [5]:
for experiment in EXPERIMENTS:
    start, end = get_period_for_experiment(experiment)

    input_dir = (
        BASE_INPUT_DIR
        / MODEL
        / experiment
        / VARIABLE
    )

    output_dir = (
        BASE_OUTPUT_DIR
        / VARIABLE_LONG_NAME
        / COUNTRY_FOLDER
        / MODEL
        / experiment
    )

    temp_dir = (
        BASE_TEMP_DIR
        / VARIABLE_LONG_NAME
        / COUNTRY_FOLDER
        / MODEL
        / experiment
    )

    step1_file = temp_dir / f"{VARIABLE}_step1_country_subset.nc"
    output_file = output_dir / f"{VARIABLE}_{start}_{end}_processed.nc"

    print("" + "=" * 80)
    print(f"Experiment: {experiment}")
    print(f"Input folder: {input_dir}")
    print(f"Output file: {output_file}")
    print("=" * 80)

    ds = load_cmip6_data(input_dir=input_dir, experiment=experiment)

    ds = cut_dataset_to_area(
        ds=ds,
        area=AREA,
        buffer_degrees=SPATIAL_BUFFER_DEGREES,
    )

    save_netcdf(ds=ds, output_filename=step1_file, overwrite=OVERWRITE)

    # Reopen the temporary file before the temporal aggregation.
    # This keeps memory use lower for long scenario time series.
    ds = xr.open_dataset(step1_file)

    if AGGREGATION_MODE == "daymean":
        ds = aggregate_daily_mean(ds, variable_name=VARIABLE)
        print("Data aggregated by daily mean.")
    elif AGGREGATION_MODE is None:
        print("No temporal aggregation applied.")
    else:
        raise ValueError(
            f"Aggregation mode '{AGGREGATION_MODE}' is not supported. "
            "Use 'daymean' or None."
        )

    save_netcdf(ds=ds, output_filename=output_file, overwrite=OVERWRITE)


Experiment: ssp585
Input folder: ../data/esgf_downloads/CNRM-ESM2-1/ssp585/rivo
Output file: ../data/processed/river_discharge/bolivia/CNRM-ESM2-1/ssp585/rivo_2015-01_2100-12_processed.nc
Input files found: 2
First file: rivo_Eday_CNRM-ESM2-1_ssp585_r1i1p1f2_gn_20150101-20641231.nc
Data written to: ../data/processed/river_discharge/bolivia/CNRM-ESM2-1/ssp585/rivo_step1_country_subset.nc
Data aggregated by daily mean.
Data written to: ../data/processed/river_discharge/bolivia/CNRM-ESM2-1/ssp585/rivo_2015-01_2100-12_processed.nc


## Step 6. Check the processed outputs

This step lists the processed NetCDF files created by the notebook.


In [6]:
for experiment in EXPERIMENTS:
    start, end = get_period_for_experiment(experiment)

    output_file = (
        BASE_OUTPUT_DIR
        / VARIABLE_LONG_NAME
        / COUNTRY_FOLDER
        / experiment
        / MODEL
        / f"{VARIABLE}_{start}_{end}_processed.nc"
    )

    print("" + "=" * 80)
    print(f"Experiment: {experiment}")
    print(f"Processed file: {output_file}")
    print(f"Exists: {output_file.exists()}")

    if output_file.exists():
        with xr.open_dataset(output_file) as ds_check:
            print(ds_check)


Experiment: ssp585
Processed file: ../data/processed/river_discharge/bolivia/ssp585/CNRM-ESM2-1/rivo_2015-01_2100-12_processed.nc
Exists: False


## Notes for the next processing steps

The processed CMIP6 files are saved in:

`../data/processed/{VARIABLE_LONG_NAME}/{COUNTRY_FOLDER}/{MODEL}/{experiment}/`

For example:

`../data/processed/river_discharge/ecuador/CNRM-ESM2-1/ssp370/rivo_2015-01_2100-12_processed.nc`

These files can be used as input for the basin statistics workflow.
